# Module 21 — Flask

The first of five presentations of the same data. `app.py` next to this notebook is
the finished version; this notebook builds it up a piece at a time.

Two things worth having in mind from the start.

**A Flask application is a set of functions with URLs attached**, and the attaching is
`@app.get("/")` — a decorator, which module 14 explained: it registers the function
somewhere and hands it back unchanged. Nothing about the route is magic, and you have
already read the mechanism.

**And you do not need a server to test it.** `app.test_client()` sends requests
straight into the application, without a socket, a port or a running process. Every
cell below uses it, which is why this notebook can be a notebook at all.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "app.py").is_file() else Path.cwd() / "21_flask"
sys.path.append(str(HERE))  # so that `import app` finds it -- module 10

from flask import Flask  # noqa: E402

from sensorreport import LIMIT, load_readings, summarise  # noqa: E402

print(len(load_readings()), "readings, limit", LIMIT)

## 1. A route is a decorated function

`Flask(__name__)` makes the application; `@app.get(path)` attaches a function to a
URL. The function's **return value becomes the response body**.

In [ ]:
app = Flask(__name__)


@app.get("/")
def index():
    return "<h1>Sensors</h1>"


with app.test_client() as client:
    response = client.get("/")
    print(response.status_code, response.headers["Content-Type"])
    print(response.get_data(as_text=True))

No port, no process, no `app.run()`. `test_client()` is the whole of what module 15
would call a unit under test — and it is how Flask applications are actually tested,
which is why this is worth learning before the server.

Three things a view can return, and Flask does something different with each:

In [ ]:
app = Flask(__name__)


@app.get("/text")
def as_text():
    return "just words"  # str -> text/html


@app.get("/json")
def as_json():
    return {"tag": "TH-04", "value": 91.0}  # dict -> application/json


@app.get("/created")
def with_status():
    return "made it", 201  # a tuple -> body and status code


with app.test_client() as client:
    for path in ("/text", "/json", "/created"):
        response = client.get(path)
        print(f"{path:10} {response.status_code} {response.headers['Content-Type']}")

A returned `dict` becoming JSON is the shortcut that makes a small API two lines — and
section 6 uses it. Module 23 is what you reach for when the API is the point rather
than an afterthought.

## 2. Reading the request

Two places a value can come from, and they are handled differently.

**Part of the path** — `/location/Hall` — becomes a function argument, declared in the
route with `<name>`.

In [ ]:
app = Flask(__name__)


@app.get("/location/<name>")
def location(name):
    return {"asked_for": name, "type": type(name).__name__}


@app.get("/reading/<int:number>")  # a converter: only matches digits, and converts
def reading(number):
    return {"number": number, "type": type(number).__name__}


with app.test_client() as client:
    print(client.get("/location/Hall").get_json())
    print(client.get("/reading/5").get_json())
    print("/reading/abc ->", client.get("/reading/abc").status_code, "-- the route did not match")

`<int:number>` is a **converter**: it only matches digits and hands you an `int`. A
path that does not match is not an error in your function — it is a 404, because no
route claimed it. Which is better than a `ValueError` inside a view, and is worth
choosing deliberately.

**Query string** — `/search?above=85` — comes from `request.args`, which is a dict.

In [ ]:
from flask import request

app = Flask(__name__)


@app.get("/search")
def search():
    return {
        "raw": request.args.get("above"),  # always a string, or None
        "converted": request.args.get("above", type=float),  # or None if it will not convert
        "with_default": request.args.get("above", default=85.0, type=float),
    }


with app.test_client() as client:
    print(client.get("/search?above=91.5").get_json())
    print(client.get("/search").get_json())
    print(client.get("/search?above=lots").get_json())

Read the last line. `type=float` on a value that will not convert gives **`None`**, not
an exception — which is the right behaviour for something a person typed into a box,
and a decision you have to notice: your view has to handle `None`, and forgetting to
is how a search page returns a 500 to somebody who typed a word.

Everything from a URL is a string, exactly as in modules 08, 17 and 18. Four modules,
one lesson.

## 3. Templates

Returning HTML from a Python string works for one heading and collapses immediately
after. Jinja2 templates live in `templates/` and are rendered with
`render_template`.

In [ ]:
from flask import render_template_string

app = Flask(__name__)

TEMPLATE = """
<table>
  {% for s in summaries %}
  <tr><td>{{ s.location }}</td><td>{{ s.usable }}</td><td>{{ s.mean }}</td></tr>
  {% endfor %}
</table>
"""

with app.app_context():  # render_template_string needs one; a real request has it already
    print(render_template_string(TEMPLATE, summaries=summarise(load_readings())).strip())

`{{ ... }}` prints, `{% ... %}` controls. That is the whole syntax, and the rest is
filters — `|length`, `|round(1)`, `|format`, `|selectattr` — which are functions
written after a pipe.

Two Jinja2 details that are worth measuring rather than assuming.

**The `default` filter does not replace `None`.** It replaces an *undefined* value,
and `None` is perfectly defined. This catches everybody once:

In [ ]:
from flask import render_template_string

app = Flask(__name__)

with app.app_context():
    with_filter = render_template_string("{{ v|default('--') }}", v=None)
    with_test = render_template_string("{{ v if v is not none else '--' }}", v=None)

# One of these prints '--'. Which, and what does the other print?
assert with_filter == ...
assert with_test == ...

So for a value that may be `None` — like a mean over no readings — the test is
`{{ x if x is not none else "--" }}`. `|default` is for a variable that might not have
been passed at all.

**And Jinja2 escapes HTML by default**, which is the reason a template is safer than a
Python f-string.

In [ ]:
from flask import render_template_string

app = Flask(__name__)
hostile = "<script>alert('xss')</script>"

with app.app_context():
    print("escaped: ", render_template_string("<p>{{ text }}</p>", text=hostile))
    print("with |safe:", render_template_string("<p>{{ text|safe }}</p>", text=hostile))

The first is inert text in the page. The second is a script the browser runs —
**cross-site scripting**, and the shape is module 19's SQL injection exactly: text
from outside became syntax on the inside, because it was pasted in rather than passed
in.

`|safe` says "I know this is markup and I trust it". Applying it to anything a user
supplied is the bug. And building HTML with an f-string is `|safe` for the whole
string, which is why templates are not merely tidier.

## 4. `url_for`, and never joining strings

A link written as `"/location/" + name` breaks in three ways: it does not escape the
name, it hard-codes the path, and it is wrong the day the route changes. `url_for`
takes the **name of the view function** and builds the URL from the route.

In [ ]:
from flask import url_for

app = Flask(__name__)


# ruff: noqa: F811 -- each cell builds a fresh app, so the view names repeat.
# ruff analyses the whole notebook as one file and is right about the letter of it.
@app.get("/location/<name>")
def location(name):
    return name


@app.get("/search")
def search():
    return "search"


with app.test_request_context():
    print(url_for("location", name="Hall"))
    print(url_for("location", name="Test rig"))  # the space is encoded for you
    print(url_for("search", above=85))  # an unknown argument becomes a query parameter

`Test rig` came out as `Test%20rig` — the encoding of module 16, done for you. And
`above=85` became `?above=85` because `search` has no `<above>` in its route, so
Flask put it in the query string.

Which is the argument in one line: **`url_for` names the function, so the URL is
allowed to change.** Rename the path in the decorator and every link still works.

## 5. Status codes, and what `abort` is for

A view that cannot answer should say so with a status code, not with an empty page or
a traceback. `abort(404)` raises, so nothing after it runs, and Flask turns it into a
proper response — module 16's four hundred and four, from the other end.

In [ ]:
from flask import abort

app = Flask(__name__)


@app.get("/location/<name>")
def location(name):
    known = {s.location for s in summarise(load_readings())}
    if name not in known:
        abort(404)
    return {"location": name}


with app.test_client() as client:
    print(client.get("/location/Hall").status_code, client.get("/location/Hall").get_json())
    print(client.get("/location/Nowhere").status_code)
    print(client.get("/no/such/route").status_code, "-- no route matched, so Flask says 404 itself")

## 6. The whole application

`app.py` next to this notebook is all of the above put together: four routes, four
templates, and every number from `sensorreport`. Import it and look at what it
answers.

In [ ]:
import app as flask_app

with flask_app.app.test_client() as client:
    for path in ("/", "/location/Hall", "/location/Nowhere", "/search?above=85", "/api/summary"):
        response = client.get(path)
        kind = response.headers["Content-Type"].split(";")[0]
        print(f"{path:22} {response.status_code} {kind:18} {len(response.get_data()):>5} bytes")

In [ ]:
import app as flask_app

with flask_app.app.test_client() as client:
    payload = client.get("/api/summary").get_json()

print(payload["limit"])
for entry in payload["locations"]:
    print(f"  {entry['location']:<10}{entry['usable']:>3}{entry['mean']:>8}{entry['faults']:>3}")

Those are the numbers from modules 18, 19 and 20 for the fourth time, and the point is
that this module contributed none of them. `app.py` is one import and four view
functions; the analysis is `sensorreport`, unchanged.

To see it in a browser:

```console
uv run flask --app 21_flask/app run --debug
```

`--debug` gives you the reloader and the traceback page. Module 20 said what that page
also gives anybody who can reach the port, so it stays on `127.0.0.1`, which is the
default.

## 7. What Flask is, and what it is not

Flask is a **micro**framework, and the word is accurate rather than modest: routing,
templates, request parsing, and a development server. That is the list.

What it does not include, and what you reach for when you need it:

| | |
| --- | --- |
| a database layer | SQLAlchemy, or module 19's `sqlite3` directly |
| forms and validation | WTForms |
| users and login | Flask-Login |
| an admin interface | write one, or Django |
| migrations | Alembic |

That is the trade, and it goes both ways. For this application — read some data, show
a table — Flask is about seventy lines including the templates, and there is nothing
in it you did not choose. For an application with users, permissions, an admin
interface and thirty models, you would be assembling from parts what **Django** ships
assembled, and Django would be the better answer.

The heuristic: **Flask when you can name everything your application needs; Django
when you would rather not have to.**

---

`exercises/` is next: seven files to fill in and two to think through. All of them use
`test_client()`, so none of them needs a port.

Module 22 is Streamlit — the same numbers with no routes, no templates and no HTML at
all, which makes it the sharpest comparison in Part 5.